In [1]:
import json
import statistics
from uuid import uuid4
from pathlib import Path
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()


C:\2026-Projects\Document_Intelligent_Hub\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
PROJECT_ROOT = Path.cwd()

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

INGESTED_DOCUMENTS_PATH = PROCESSED_DATA_DIR / "ingested_documents.json"
CHUNKED_DOCUMENTS_PATH =PROCESSED_DATA_DIR / "chunked_documents.json"

print(f"Ingested input path: {INGESTED_DOCUMENTS_PATH}")
print(f"Chunked output path: {CHUNKED_DOCUMENTS_PATH}")

Ingested input path: C:\document_hub\notebooks\data\processed\ingested_documents.json
Chunked output path: C:\document_hub\notebooks\data\processed\chunked_documents.json


In [3]:
def json_records_to_documents(records: list[dict]) -> list[Document]:
    documents = []

    for record in records:
        documents.append(
            Document(
                page_content=record["page_content"],
                metadata=record["metadata"]
            )
        )

    return documents

In [4]:
def documents_to_json( documents: list[Document]) -> list[dict]:
    records = []

    for document in documents:
        records.append(
            {
                "page_content": document.page_content,
                "metadata": document.metadata
            }
        )
    return records



In [5]:
if not INGESTED_DOCUMENTS_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {INGESTED_DOCUMENTS_PATH}. "
        "Run 01_document_ingestion.ipynb first."
    )

with INGESTED_DOCUMENTS_PATH.open("r", encoding="utf-8") as file:
    ingested_records = json.load(file)

ingested_documents = json_records_to_documents(ingested_records)

print(f"Loaded {len(ingested_documents)} ingested documents.")

Loaded 3 ingested documents.


In [6]:
document_lengths = [len(document.page_content) for document in ingested_documents]

print(f"Document parts: {document_lengths}")

if document_lengths:
    print(f"Sortest document part: {min(document_lengths):,} characters")
    print(f"Longest document part: {max(document_lengths):,} characters")
    print(f"Average document part: {statistics.mean(document_lengths):,} characters")

Document parts: [1342, 1150, 869]
Sortest document part: 869 characters
Longest document part: 1,342 characters
Average document part: 1,120.3333333333333 characters


In [7]:
if ingested_documents:
    sample_document = ingested_documents[0]

    print("Metadata:")
    print(sample_document.metadata)

    print()
    print("Content preview:")
    print(sample_document.page_content[:1000])

Metadata:
{'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-06-20T21:35:22+02:00', 'author': 'BENNET DYANI', 'moddate': '2026-06-20T21:35:22+02:00', 'source': 'Sample Compliance Manual.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'document_id': 'a7777030-0f2d-4a90-a97e-c92105894b71', 'file_path': 'C:\\document_hub\\notebooks\\data\\raw\\Sample Compliance Manual.pdf', 'file_type': 'pdf', 'department': 'Compliance', 'access_role': 'Compliance Analyst', 'ingested_at': '2026-06-26T16:46:03.274307+00:00', 'page_number': 1, 'part_index': 0}

Content preview:
Sample Compliance Manual 
 
Page 1 — Introduction & Scope 
This Compliance Manual sets forth the standards governing Anti-Money Laundering 
(AML), Data Privacy, and Operational Risk Management across the enterprise. It applies 
to all employees, contractors, and third-party vendors engaged with the organization. 
Compliance is not optional. Failure to adhere t

In [8]:
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150

In [9]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size= CHUNK_SIZE,
    chunk_overlap= CHUNK_OVERLAP,
    separators=["\n\n", "\n", " ", ""]
)

chunked_documents = text_splitter.split_documents(ingested_documents)

print(f"Original document parts: {len(ingested_documents)}")
print(f"Chunked document: {len(chunked_documents)}")

Original document parts: 3
Chunked document: 5


In [10]:

def enrich_chunk_metadata(chunks: list[Document]) -> list[Document]:
    enriched_chunks = []

    for global_chunk_index, chunk in enumerate(chunks):
        source = chunk.metadata.get("source", "Unknown_source")
        document_id = chunk.metadata.get("document_id", "unknown_document")

        chunk.metadata.update(
            {
                "chunk_id": str(uuid4()),
                "chunk_index": global_chunk_index,
                "chunk_size": len(chunk.page_content),
                "document_id": document_id,
                "source": source,
            }
        )

        enriched_chunks.append(chunk)
    return enriched_chunks

In [11]:
chunked_documents = enrich_chunk_metadata(chunked_documents)
print(f"Chunked document parts: {len(chunked_documents)}")


Chunked document parts: 5


In [12]:
if chunked_documents:
    first_chunk = chunked_documents[0]

    print("Chunk metadata:")
    print(first_chunk.metadata)

    print()
    print("Chunk content preview:")
    print(first_chunk.page_content)


Chunk metadata:
{'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-06-20T21:35:22+02:00', 'author': 'BENNET DYANI', 'moddate': '2026-06-20T21:35:22+02:00', 'source': 'Sample Compliance Manual.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'document_id': 'a7777030-0f2d-4a90-a97e-c92105894b71', 'file_path': 'C:\\document_hub\\notebooks\\data\\raw\\Sample Compliance Manual.pdf', 'file_type': 'pdf', 'department': 'Compliance', 'access_role': 'Compliance Analyst', 'ingested_at': '2026-06-26T16:46:03.274307+00:00', 'page_number': 1, 'part_index': 0, 'chunk_id': '5b8ab536-fd07-45ad-9e2e-fba16b375fbd', 'chunk_index': 0, 'chunk_size': 991}

Chunk content preview:
Sample Compliance Manual 
 
Page 1 — Introduction & Scope 
This Compliance Manual sets forth the standards governing Anti-Money Laundering 
(AML), Data Privacy, and Operational Risk Management across the enterprise. It applies 
to all employees, contractors, an

In [13]:
chunk_lengths = [len(chunk.page_content) for chunk in chunked_documents]

print(f"Total chunk parts: {len(chunked_documents)}")

Total chunk parts: 5


In [14]:
if chunk_lengths:
    print(f"Shortest chunk part: {min(chunk_lengths):,} characters")
    print(f"Longest chunk part: {max(chunk_lengths):,} characters")
    print(f"Average chunk part: {statistics.mean(chunk_lengths):,} characters")


Shortest chunk part: 301 characters
Longest chunk part: 991 characters
Average chunk part: 722.8 characters


In [15]:
chunks_by_source ={}

for chunk in chunked_documents:
    source = chunk.metadata.get("source", "unknown_source")
    chunks_by_source[source] = chunks_by_source.get(source, 0) + 1

for source, count in chunks_by_source.items():
    print(f"{source}: {count:,} chunks")

Sample Compliance Manual.pdf: 5 chunks


In [16]:
chunked_records = documents_to_json(chunked_documents)

with CHUNKED_DOCUMENTS_PATH.open("w", encoding="utf-8") as file:
    json.dump(chunked_records, file, indent=2)

print(f"Saved {len(chunked_records)} chunked documents to {CHUNKED_DOCUMENTS_PATH}")


with CHUNKED_DOCUMENTS_PATH.open("r", encoding="utf-8") as file:
    reloaded_records = json.load(file)

reloaded_chunks = json_records_to_documents(reloaded_records)

print(f"Reloaded {len(reloaded_chunks)} chunks.")

if reloaded_chunks:
    print(reloaded_chunks[0].metadata)
    print()
    print(reloaded_chunks[0].page_content[:500])

Saved 5 chunked documents to C:\document_hub\notebooks\data\processed\chunked_documents.json
Reloaded 5 chunks.
{'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-06-20T21:35:22+02:00', 'author': 'BENNET DYANI', 'moddate': '2026-06-20T21:35:22+02:00', 'source': 'Sample Compliance Manual.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'document_id': 'a7777030-0f2d-4a90-a97e-c92105894b71', 'file_path': 'C:\\document_hub\\notebooks\\data\\raw\\Sample Compliance Manual.pdf', 'file_type': 'pdf', 'department': 'Compliance', 'access_role': 'Compliance Analyst', 'ingested_at': '2026-06-26T16:46:03.274307+00:00', 'page_number': 1, 'part_index': 0, 'chunk_id': '5b8ab536-fd07-45ad-9e2e-fba16b375fbd', 'chunk_index': 0, 'chunk_size': 991}

Sample Compliance Manual 
 
Page 1 — Introduction & Scope 
This Compliance Manual sets forth the standards governing Anti-Money Laundering 
(AML), Data Privacy, and Operational Risk Manage